In [1]:
import pandas as pd

courses = pd.read_csv("https://waf.cs.illinois.edu/discovery/course-catalog.csv")
print (courses.columns)
print (courses.head())

Index(['Year', 'Term', 'YearTerm', 'Subject', 'Number', 'Name', 'Description',
       'Credit Hours', 'Section Info', 'Degree Attributes',
       'Schedule Information', 'CRN', 'Section', 'Status Code', 'Part of Term',
       'Section Title', 'Section Credit Hours', 'Section Status',
       'Enrollment Status', 'Type', 'Type Code', 'Start Time', 'End Time',
       'Days of Week', 'Room', 'Building', 'Instructors'],
      dtype='str')
   Year    Term YearTerm Subject  Number                          Name  \
0  2026  Spring  2026-sp     AAS     100  Intro Asian American Studies   
1  2026  Spring  2026-sp     AAS     100  Intro Asian American Studies   
2  2026  Spring  2026-sp     AAS     100  Intro Asian American Studies   
3  2026  Spring  2026-sp     AAS     100  Intro Asian American Studies   
4  2026  Spring  2026-sp     AAS     100  Intro Asian American Studies   

                                         Description Credit Hours  \
0  Interdisciplinary introduction to the basic c

In [2]:
courses["Credit Hours Clean"] = courses["Credit Hours"].str.extract(r"(\d+)").astype(float)

In [3]:
courses["Start Time Clean"] = pd.to_datetime(
    courses["Start Time"], format="%I:%M %p", errors="coerce"
).dt.time

courses["End Time Clean"] = pd.to_datetime(
    courses["End Time"], format="%I:%M %p", errors="coerce"
).dt.time
print (courses[["Credit Hours", "Credit Hours Clean", "Start Time", "Start Time Clean", "End Time", "End Time Clean"]].head())

  Credit Hours  Credit Hours Clean Start Time Start Time Clean  End Time  \
0     3 hours.                 3.0   09:00 AM         09:00:00  09:50 AM   
1     3 hours.                 3.0   10:00 AM         10:00:00  10:50 AM   
2     3 hours.                 3.0   11:00 AM         11:00:00  11:50 AM   
3     3 hours.                 3.0   12:00 PM         12:00:00  12:50 PM   
4     3 hours.                 3.0   01:00 PM         13:00:00  01:50 PM   

  End Time Clean  
0       09:50:00  
1       10:50:00  
2       11:50:00  
3       12:50:00  
4       13:50:00  


In [4]:
import pandas as pd

RAW_TO_SIMPLE_TYPE = {
    "Lecture": "Lecture",
    "Online Lecture": "Lecture",
    "Lecture-Discussion": "Lecture-Discussion",
    "Online Lecture Discussion": "Lecture-Discussion",

    "Discussion/Recitation": "Discussion",
    "Online Discussion": "Discussion",

    "Laboratory": "Lab",
    "Online Lab": "Lab",
    "Laboratory-Discussion": "Lab",

    "Online": "Online course",

    "Independent Study": None,
    "Conference": None,
    "Seminar": None,
    "Study Abroad": None,
    "Internship": None,
    "Research": None,
    "Travel": None,
    "Studio": None,
    "Practice": None,
    "Quiz": None,
    "Packaged Section": None,
}

def normalize_type(type_value):
    if pd.isna(type_value):
        return None
    return RAW_TO_SIMPLE_TYPE.get(str(type_value).strip(), None)


In [5]:
def row_matches_schedule(row, days=None, start_time=None, end_time=None):
    """
    Check whether ONE section (row) fits the user's schedule constraints.
    Returns True if it fits, False if it doesn't.
    """
    # Days filter
    if days:
        wanted_days = set(days)
        row_days = set(str(row.get("Days of Week", "")).strip())
        if not row_days:
            return False
        # Class days must be a subset of user's available days
        # e.g. user says MWF, class is MW → allowed
        if not row_days.issubset(wanted_days):
            return False

    # Time filter: class must fit fully within the user's window
    row_start = row.get("Start Time Clean")
    row_end = row.get("End Time Clean")

    if start_time is not None and row_start is not None:
        if row_start < start_time:
            return False

    if end_time is not None and row_end is not None:
        if row_end > end_time:
            return False

    return True


def course_matches_bundle(course_df, days=None, start_time=None, end_time=None):
    """
    Keep a course only if:
    1. All its section types are supported (no unsupported types like Quiz, Studio)
    2. Every required component (lecture / discussion / lab) has at least ONE
       section that fits the user's days/time window
    """
    course_df = course_df.copy()
    course_df["Simple Type"] = course_df["Type"].apply(normalize_type)

    # Reject the whole course if it contains any unsupported type
    if course_df["Simple Type"].isna().any():
        return False

    # Split rows by simplified type
    lecture_rows    = course_df[course_df["Simple Type"] == "Lecture"]
    lec_dis_rows    = course_df[course_df["Simple Type"] == "Lecture-Discussion"]
    discussion_rows = course_df[course_df["Simple Type"] == "Discussion"]
    lab_rows        = course_df[course_df["Simple Type"] == "Lab"]
    online_rows     = course_df[course_df["Simple Type"] == "Online course"]

    def has_valid(part_df):
        """Does this component have at least one section that fits the schedule?"""
        return any(
            row_matches_schedule(row, days=days, start_time=start_time, end_time=end_time)
            for _, row in part_df.iterrows()
        )

    # Case 1: Online-only course
    if lecture_rows.empty and lec_dis_rows.empty and discussion_rows.empty and lab_rows.empty:
        return has_valid(online_rows)

    # Case 2: Lecture-Discussion combined type
    if not lec_dis_rows.empty:
        if not has_valid(lec_dis_rows):
            return False
        if not discussion_rows.empty and not has_valid(discussion_rows):
            return False
        if not lab_rows.empty and not has_valid(lab_rows):
            return False
        return True

    # Case 3: Normal structure — Lecture + optional Discussion + optional Lab
    if not lecture_rows.empty and not has_valid(lecture_rows):
        return False
    if not discussion_rows.empty and not has_valid(discussion_rows):
        return False
    if not lab_rows.empty and not has_valid(lab_rows):
        return False

    return True


def filter_courses(
    df,
    gen_ed=None,
    credits=None,
    days=None,
    part_of_term=None,
    start_time=None,
    end_time=None,
):
    """
    Main filtering function.
    Step 1: Apply simple row-level filters (gen-ed, credits, part of term).
    Step 2: Group by Subject + Number and validate each full course bundle.
    """
    result = df.copy()

    if gen_ed:
        result = result[
            result["Degree Attributes"].fillna("").str.contains(gen_ed, case=False, na=False)
        ]

    if credits is not None:
        result = result[result["Credit Hours Clean"] == credits]

    if part_of_term:
        result = result[
            result["Part of Term"].fillna("").str.contains(part_of_term, case=False, na=False)
        ]

    # Group by course and validate the whole bundle
    filtered = result.groupby(["Subject", "Number"]).filter(
        lambda course: course_matches_bundle(
            course,
            days=days,
            start_time=start_time,
            end_time=end_time,
        )
    )

    return filtered


In [6]:
from datetime import time

filtered = filter_courses(
    courses,
    gen_ed="Humanities",
    credits=3,
    days=["M", "W"],
    part_of_term="1",
    start_time=time(10, 0),
    end_time=time(16, 0)
)

print(filtered[[
    "Subject", "Number", "Name",
    "Credit Hours Clean", "Days of Week",
    "Start Time", "End Time", "Degree Attributes",
    "Type", "Instructors", "Building", "Room"
]].head(20))


     Subject  Number                                        Name  \
613     AFRO     224               Humanist Persp of Afro-Am Exp   
618     AFRO     261               Intro to the African Diaspora   
685      AIS     101            Intro to American Indian Studies   
691      AIS     285                         Indigenous Thinkers   
692      AIS     285                         Indigenous Thinkers   
693      AIS     285                         Indigenous Thinkers   
720     ANSC     120  The Art and Ethics of Animal Documentaries   
877     ANTH     261               Intro to the African Diaspora   
1210    ARTH     220               African Arts and Architecture   
3801      CW     100                   Intro to Creative Writing   
3802      CW     100                   Intro to Creative Writing   
3803      CW     100                   Intro to Creative Writing   
3804      CW     100                   Intro to Creative Writing   
3805      CW     100                   Intro to 

In [7]:
from datetime import time

# ── Sample data to test the bundle filtering logic ──────────────────────────
test_data = {
    "Subject":  ["HIST", "HIST", "HIST",   "MATH", "MATH",   "PHYS", "PHYS", "PHYS",   "ART",  "ART"],
    "Number":   [101,    101,    101,       201,    201,      301,    301,    301,       401,    401],
    "Name":     ["World History"]*3 + ["Calculus"]*2 + ["Physics"]*3 + ["Studio Art"]*2,
    "Type":     ["Lecture", "Discussion/Recitation", "Discussion/Recitation",
                 "Lecture", "Laboratory",
                 "Lecture", "Discussion/Recitation", "Quiz",
                 "Lecture", "Studio"],
    "Days of Week": ["MWF", "T",  "R",    "TR",  "T",     "MWF", "T",  "W",   "MWF", "F"],
    "Start Time Clean": [time(10,0), time(10,0), time(11,0),   time(9,0),  time(14,0),
                         time(10,0), time(10,0), time(20,0),   time(10,0), time(10,0)],
    "End Time Clean":   [time(10,50),time(10,50),time(11,50),  time(9,50), time(14,50),
                         time(10,50),time(10,50),time(20,50),  time(10,50),time(10,50)],
    "Degree Attributes": ["Humanities"]*10,
    "Credit Hours Clean": [3]*10,
    "Part of Term": ["1"]*10,
    "Instructors": ["Smith, J"]*10,
    "Building": ["Lincoln Hall"]*10,
    "Room": ["100"]*10,
}

test_df = pd.DataFrame(test_data)

result = filter_courses(
    test_df,
    days=["M", "T", "W", "R", "F"],
    start_time=time(9, 0),
    end_time=time(17, 0),
)

passed = set(zip(result["Subject"], result["Number"]))

print("=== TEST RESULTS ===\n")

tests = [
    ("HIST", 101, True,  "Lecture MWF + two discussions (T, R) — all fit → should PASS"),
    ("MATH", 201, True,  "Lecture TR + Lab T 2pm — all fit → should PASS"),
    ("PHYS", 301, False, "Has a Quiz section → unsupported type → should FAIL"),
    ("ART",  401, False, "Has a Studio section → unsupported type → should FAIL"),
]

all_passed = True
for subj, num, expected, description in tests:
    actual = (subj, num) in passed
    status = "✅ PASS" if actual == expected else "❌ FAIL"
    if actual != expected:
        all_passed = False
    print(f"{status}  {subj} {num}: {description}")

print()
print("All tests passed! ✅" if all_passed else "Some tests failed ❌ — check logic above.")


=== TEST RESULTS ===

✅ PASS  HIST 101: Lecture MWF + two discussions (T, R) — all fit → should PASS
✅ PASS  MATH 201: Lecture TR + Lab T 2pm — all fit → should PASS
✅ PASS  PHYS 301: Has a Quiz section → unsupported type → should FAIL
✅ PASS  ART 401: Has a Studio section → unsupported type → should FAIL

All tests passed! ✅
